In [ ]:
%%capture
!pip install transformers torch rdkit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

DRIVE_OUT = '/content/drive/MyDrive/DDI_project/coverage_analysis/'
print('Ready.')

Mounted at /content/drive
Ready.


In [ ]:
coverage = pd.read_csv(DRIVE_OUT + 'ogb_ddi_full_coverage_table.csv')
print(f'Coverage table: {coverage.shape}')
print(f'Valid SMILES: {coverage["smiles_rdkit_valid"].sum()}')
print(f'Missing SMILES: {(~coverage["smiles_rdkit_valid"]).sum()}')

Coverage table: (4267, 6)
Valid SMILES: 3543
Missing SMILES: 724


In [ ]:
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
model.to('cuda')
print('ChemBERTa loaded on', next(model.parameters()).device)

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChemBERTa loaded on cuda:0


In [ ]:
def embed_batch(texts, tokenizer, model, max_length=512):
    inputs = tokenizer(
        texts,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
        padding=True
    ).to('cuda')
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu().numpy()

In [ ]:
valid = coverage[coverage['smiles_rdkit_valid'] == True].copy()
print(f'Generating ChemBERTa embeddings for {len(valid)} drugs...')

all_embeddings = []
batch_size = 32

for i in range(0, len(valid), batch_size):
    batch = valid.iloc[i:i+batch_size]['smiles'].tolist()
    embs = embed_batch(batch, tokenizer, model)
    all_embeddings.append(embs)
    if i % 500 == 0:
        print(f'  {i} / {len(valid)} done')

embeddings_valid = np.vstack(all_embeddings)
drug_ids_valid = valid['drugbank_id'].tolist()
print(f'Done. Shape: {embeddings_valid.shape}')

Generating ChemBERTa embeddings for 3543 drugs...
  0 / 3543 done
Done. Shape: (3543, 768)


In [ ]:
mean_vector = embeddings_valid.mean(axis=0)

emb_lookup = dict(zip(drug_ids_valid, embeddings_valid))
all_drug_ids = coverage['drugbank_id'].tolist()

full_embeddings = np.array([
    emb_lookup.get(did, mean_vector) for did in all_drug_ids
])

n_fallback = sum(did not in emb_lookup for did in all_drug_ids)
print(f'Full matrix shape: {full_embeddings.shape}')
print(f'Drugs using mean fallback: {n_fallback}')

Full matrix shape: (4267, 768)
Drugs using mean fallback: 724


In [ ]:
mean_vector = embeddings_valid.mean(axis=0)

emb_lookup = dict(zip(drug_ids_valid, embeddings_valid))
all_drug_ids = coverage['drugbank_id'].tolist()

full_embeddings = np.array([
    emb_lookup.get(did, mean_vector) for did in all_drug_ids
])

n_fallback = sum(did not in emb_lookup for did in all_drug_ids)
print(f'Full matrix shape: {full_embeddings.shape}')
print(f'Drugs using mean fallback: {n_fallback}')

Full matrix shape: (4267, 768)
Drugs using mean fallback: 724


In [ ]:
print(f'Any NaN: {np.isnan(full_embeddings).any()}')
print(f'Any all-zero rows: {(full_embeddings == 0).all(axis=1).sum()}')
print(f'Embedding norm (first drug): {np.linalg.norm(full_embeddings[0]):.4f}')

# Cosine similarity between two known similar drugs (both statins)
emb_dict = dict(zip(all_drug_ids, full_embeddings))

pairs = [
    ('DB00641', 'DB00734'),  # Simvastatin, Pravastatin (both statins)
    ('DB00001', 'DB01109'),  # Lepirudin, Heparin (both anticoagulants)
]

for id1, id2 in pairs:
    if id1 in emb_dict and id2 in emb_dict:
        sim = cosine_similarity(
            emb_dict[id1].reshape(1,-1),
            emb_dict[id2].reshape(1,-1)
        )[0][0]
        print(f'Cosine similarity {id1} vs {id2}: {sim:.4f}')
    else:
        print(f'One of {id1}, {id2} not found in embeddings')


Any NaN: False
Any all-zero rows: 0
Embedding norm (first drug): 21.6966
Cosine similarity DB00641 vs DB00734: 0.3943
Cosine similarity DB00001 vs DB01109: 1.0000


In [ ]:
np.save(DRIVE_OUT + 'chemberta_embeddings.npy', full_embeddings)
np.save(DRIVE_OUT + 'chemberta_drug_ids.npy', np.array(all_drug_ids))
np.save(DRIVE_OUT + 'chemberta_embeddings_valid_only.npy', embeddings_valid)
np.save(DRIVE_OUT + 'chemberta_drug_ids_valid_only.npy', np.array(drug_ids_valid))

print(f'Saved full matrix:       {full_embeddings.shape}')
print(f'Saved valid-only matrix: {embeddings_valid.shape}')

Saved full matrix:       (4267, 768)
Saved valid-only matrix: (3543, 768)


In [ ]:
# Option 1 — Concatenation (most common)
pair_feature = np.concatenate([emb_A, emb_B])  # 768+768 = 1536-dim

# Option 2 — Element-wise product (Hadamard)
pair_feature = emb_A * emb_B  # 768-dim

# Option 3 — Element-wise absolute difference
pair_feature = np.abs(emb_A - emb_B)  # 768-dim

NameError: name 'emb_A' is not defined